In [1]:
import pandas as pd
from pathlib import Path
import geopandas as gpd

In [2]:
PROJECT_DIR = Path("/home/sc4r3cr0w/Schreibtisch/Data Analytics/WBS_Final_Project")
RAW = PROJECT_DIR / "RAW"
CLEAN_DIR = PROJECT_DIR / "CLEAN"
CLEAN_DIR.mkdir(exist_ok=True)

CHARGING_DIR = RAW / "charging_points"
BOUNDARIES_DIR = RAW / "area_boundaries"

for folder in [CHARGING_DIR, BOUNDARIES_DIR]:
    print(f"{folder.name:20} exists: {folder.exists()}  files: {len(list(folder.glob('*')))}")

charging_points      exists: True  files: 1
area_boundaries      exists: True  files: 1


In [3]:
register = pd.read_csv(
    CHARGING_DIR / "Ladesaeulenregister_BNetzA_2026-09-01.csv",
    sep=";",
    encoding="utf-8-sig",
    skiprows=10,
    dtype=str,
)
print(register.shape)
register[["Kreis/kreisfreie Stadt", "Inbetriebnahmedatum", "Anzahl Ladepunkte"]].head()

(116443, 47)


,Kreis/kreisfreie Stadt,Inbetriebnahmedatum,Anzahl Ladepunkte
0,Landkreis Alb-Donau-Kreis,11.01.2020,2
1,Landkreis Alb-Donau-Kreis,23.11.2022,4
2,Landkreis Alb-Donau-Kreis,23.11.2022,4
3,Landkreis Alb-Donau-Kreis,23.11.2022,2
4,Landkreis Alb-Donau-Kreis,09.07.2026,1


In [4]:
charging = pd.DataFrame({
    "district_name": register["Kreis/kreisfreie Stadt"],
    "commissioning_date": pd.to_datetime(register["Inbetriebnahmedatum"], format="%d.%m.%Y"),
    "charging_points": register["Anzahl Ladepunkte"].astype(int),
})

print(charging.dtypes)
print(charging.isna().sum())
print("Earliest:", charging["commissioning_date"].min())
print("Latest:", charging["commissioning_date"].max())
print("Total charging points:", charging["charging_points"].sum())

district_name                    str
commissioning_date    datetime64[us]
charging_points                int64
dtype: object
district_name         0
commissioning_date    0
charging_points       0
dtype: int64
Earliest: 1992-10-01 00:00:00
Latest: 2026-09-01 00:00:00
Total charging points: 209136


In [6]:
# searches for the location of VG250
krs_file = list(BOUNDARIES_DIR.rglob("VG250_KRS.shp"))[0]
print(krs_file)

districts = gpd.read_file(krs_file, ignore_geometry=True)
print(districts.shape)
districts[["AGS", "BEZ", "GEN", "GF"]].head()

/home/sc4r3cr0w/Schreibtisch/Data Analytics/WBS_Final_Project/RAW/area_boundaries/vg250-ew_12-31.utm32s.shape.ebenen/vg250-ew_ebenen_1231/VG250_KRS.shp
(433, 28)


,AGS,BEZ,GEN,GF
0,01001,Kreisfreie Stadt,Flensburg,4
1,01002,Kreisfreie Stadt,Kiel,4
2,01003,Kreisfreie Stadt,Lübeck,4
3,01004,Kreisfreie Stadt,Neumünster,4
4,01051,Kreis,Dithmarschen,4


In [7]:
districts = districts[districts["GF"] == 4].copy()
districts["match_name"] = (districts["BEZ"] + " " + districts["GEN"]).str.replace(" ", "")
print("Districts:", len(districts))

charging["match_name"] = charging["district_name"].str.replace(" ", "")

charging = charging.merge(
    districts[["match_name", "AGS"]],
    on="match_name",
    how="left",
)
charging = charging.rename(columns={"AGS": "district_key"})

unmatched = charging[charging["district_key"].isna()]
print("Unmatched rows:", len(unmatched))
unmatched["district_name"].value_counts()

Districts: 400
Unmatched rows: 144


district_name
Kreisfreie Stadt Hanau    144
Name: count, dtype: int64

In [8]:
is_hanau = charging["district_name"] == "Kreisfreie Stadt Hanau"
charging.loc[is_hanau, "district_key"] = "06415"

print("Unmatched rows:", charging["district_key"].isna().sum())
print("Districts with charging points:", charging["district_key"].nunique())

Unmatched rows: 0
Districts with charging points: 401


In [9]:
frames = []

for year in range(2017, 2027):
    cutoff = pd.Timestamp(year, 1, 1)
    in_operation = charging[charging["commissioning_date"] < cutoff]
    per_district = in_operation.groupby("district_key")["charging_points"].sum().reset_index()
    per_district["year"] = year
    frames.append(per_district)

charging_yearly = pd.concat(frames, ignore_index=True)
charging_yearly = charging_yearly[["year", "district_key", "charging_points"]]

charging_yearly.groupby("year").agg(
    districts=("district_key", "size"),
    charging_points=("charging_points", "sum"),
)

,districts,charging_points
year,,
2017,287,3586
2018,357,6713
2019,390,13634
2020,399,23048
2021,401,34951
2022,401,52478
2023,401,79913
2024,401,121176
2025,401,157308


In [10]:
charging_yearly.to_csv(CLEAN_DIR / "charging_points.csv", index=False)
print("Saved:", CLEAN_DIR / "charging_points.csv", "| rows:", len(charging_yearly))

Saved: /home/sc4r3cr0w/Schreibtisch/Data Analytics/WBS_Final_Project/CLEAN/charging_points.csv | rows: 3839
